# SnowPro Advanced: Data Engineer — Exam Domain Breakdown

The table below lists the main content domains and their weightings.

| Domain | Topic | Weighting |
|--------|-------|----------|
| 1.0 | Data Movement | 28% |
| 2.0 | Performance Optimization | 19% |
| 3.0 | Storage and Data Protection | 14% |
| 4.0 | Data Governance | 14% |
| 5.0 | Data Transformation | 25% |

---

# DOMAIN 1: DATA MOVEMENT (28%)

---

## 1.1 Bulk Data Loading (COPY INTO)

### COPY INTO <table> — Advanced Options

| Option | Purpose | Default |
|--------|---------|--------|
| `ON_ERROR` | Action on load error | ABORT_STATEMENT |
| `SIZE_LIMIT` | Max bytes to load (then stops) | No limit |
| `PURGE` | Delete staged files after successful load | FALSE |
| `RETURN_FAILED_ONLY` | Only show failed files in output | FALSE |
| `MATCH_BY_COLUMN_NAME` | Map by column name not position | NONE |
| `ENFORCE_LENGTH` | Truncate or error on string overflow | TRUE |
| `TRUNCATECOLUMNS` | Truncate strings exceeding target length | FALSE |
| `FORCE` | Reload previously loaded files | FALSE |

### ON_ERROR Options

| Value | Behavior |
|-------|----------|
| `ABORT_STATEMENT` | Abort entire load on first error |
| `CONTINUE` | Skip error rows, load the rest |
| `SKIP_FILE` | Skip entire file if any error found |
| `SKIP_FILE_<n>` | Skip file if error count >= n |
| `SKIP_FILE_<n>%` | Skip file if error percentage >= n% |

### COPY INTO <location> (Unloading)

```sql
COPY INTO @my_stage/result/data_
FROM (SELECT * FROM my_table WHERE region = 'US')
FILE_FORMAT = (TYPE = 'PARQUET')
MAX_FILE_SIZE = 268435456  -- 256MB
HEADER = TRUE
OVERWRITE = TRUE;
```

**Key unloading facts:**
- Can unload to named stage, table stage, or external stage
- Supports CSV, JSON, PARQUET formats for unloading
- `SINGLE = TRUE` forces output into one file (not parallel)
- `MAX_FILE_SIZE` controls file splitting (default 16MB)
- `OVERWRITE = TRUE` replaces existing files
- Cannot unload to internal stage with encryption disabled

### Load Metadata & History

- Snowflake tracks metadata for 64 days for each file loaded via COPY
- Within 64 days: COPY skips already-loaded files automatically
- After 64 days: must use `FORCE = TRUE` or file will be loaded again
- `LOAD_HISTORY` Information Schema view: 14 days
- `COPY_HISTORY` Account Usage view: 365 days
- `VALIDATE()` function: validates staged data WITHOUT loading it

```sql
-- Validate staged files before loading
SELECT * FROM TABLE(
  VALIDATE(@my_stage/data.csv, FILE_FORMAT => 'my_csv_format')
);
```

## 1.2 Snowpipe (Continuous Loading)

### How Snowpipe Works

```
Files land in Stage → Notification → Snowpipe Queue → Serverless Load → Target Table
```

**Two triggering methods:**
1. **Auto-ingest (cloud notifications):** Cloud event triggers (S3 SQS, Azure Event Grid, GCS Pub/Sub)
2. **REST API calls:** Application calls `insertFiles` endpoint

### Auto-Ingest Setup (AWS Example)

```sql
CREATE PIPE my_pipe
  AUTO_INGEST = TRUE
  AS
  COPY INTO my_table
  FROM @my_s3_stage
  FILE_FORMAT = (TYPE = 'CSV');
```

- After creation, retrieve the SQS queue ARN from `SHOW PIPES`
- Configure S3 event notification to send to that SQS ARN

### Snowpipe Key Facts

| Aspect | Detail |
|--------|--------|
| Compute | Serverless (Snowflake-managed) |
| Billing | Per-second, based on file size |
| Load unit | Micro-batches (files) |
| Latency | Typically < 1 minute |
| File tracking | 14 days in pipe metadata |
| Max file size recommendation | 100-250 MB compressed |
| Error handling | ON_ERROR = SKIP_FILE (default for pipe) |
| Idempotency | Same file won't be loaded twice within 14 days |

### Snowpipe vs Bulk Loading

| Feature | Snowpipe | Bulk COPY |
|---------|----------|----------|
| Compute | Serverless | User warehouse |
| Trigger | Event/API | Manual/scheduled |
| Latency | Near real-time | Batch |
| Cost model | Per-file serverless credits | Warehouse credits |
| Best for | Continuous small files | Large batch loads |
| ON_ERROR default | SKIP_FILE | ABORT_STATEMENT |

### Snowpipe Streaming (Snowpipe REST API v2)

- Rows inserted via API without staging files
- Lowest latency option (seconds)
- Uses `insertRows` SDK method
- Data lands in target table without intermediate stage
- No file format needed
- Billed based on compute time for migration from row buffer to table

## 1.3 External Stages & Storage Integrations

### Storage Integration Object

```sql
CREATE STORAGE INTEGRATION my_s3_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'S3'
  STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::123456789:role/myrole'
  ENABLED = TRUE
  STORAGE_ALLOWED_LOCATIONS = ('s3://mybucket/path1/', 's3://mybucket/path2/')
  STORAGE_BLOCKED_LOCATIONS = ('s3://mybucket/path1/sensitive/');
```

**Why use Storage Integrations?**
- Avoid storing credentials in stage definitions
- Centralized access management
- Can restrict allowed/blocked locations
- Supports IAM role-based auth (no keys in Snowflake)

### Stage Types Summary

| Stage Type | Syntax | Scope | Use Case |
|-----------|--------|-------|----------|
| User stage | `@~` | Per-user | Personal files |
| Table stage | `@%table_name` | Per-table | Table-specific files |
| Named internal | `@stage_name` | Schema-level | Shared internal files |
| Named external | `@stage_name` | Schema-level | Cloud storage (S3/Azure/GCS) |

**Important distinctions:**
- User/table stages CANNOT have file formats set
- Named stages CAN have default file format
- Table stages cannot be altered or dropped (tied to table lifecycle)
- User stages cannot be dropped
- `LIST @~` shows files in your user stage
- `REMOVE @stage/path/file.csv` deletes staged files

## 1.4 Database Replication & Failover

### Replication Overview

```
Primary Account ──replication──► Secondary Account(s)
     (read/write)                    (read-only replica)
```

### Replication Group vs Failover Group

| Feature | Replication Group | Failover Group |
|---------|------------------|----------------|
| Purpose | Read-only replicas | Disaster recovery |
| Failover | NO | YES |
| Promotion | Cannot promote | Can promote secondary to primary |
| Object types | Databases, shares, etc. | Same + account-level objects |

### Objects That Can Be Replicated

- Databases (and all contained objects)
- Shares (inbound and outbound)
- Users and roles
- Warehouses
- Resource monitors
- Network policies
- Account parameters
- Integrations (certain types)

### Database Replication Key Points

```sql
-- Enable replication for a database
ALTER DATABASE my_db ENABLE REPLICATION TO ACCOUNTS org1.account2, org1.account3;

-- On secondary account: create replica
CREATE DATABASE my_db AS REPLICA OF org1.account1.my_db;

-- Refresh replica (manual)
ALTER DATABASE my_db REFRESH;
```

- Replication is asynchronous
- Secondary databases are READ-ONLY
- Refresh can be scheduled (using tasks) or manual
- Initial replication copies entire database
- Subsequent refreshes are incremental (only changes)
- Time Travel data IS replicated
- Temporary/transient tables are NOT replicated

### Failover Groups

```sql
CREATE FAILOVER GROUP my_fg
  OBJECT_TYPES = DATABASES, ROLES, USERS, WAREHOUSES
  ALLOWED_DATABASES = db1, db2
  ALLOWED_ACCOUNTS = org1.account2
  REPLICATION_SCHEDULE = '10 MINUTE';
```

- On failover, secondary becomes primary (promotion)
- Old primary becomes secondary
- `ALTER FAILOVER GROUP my_fg PRIMARY` — promotes secondary
- RPO (Recovery Point Objective): depends on replication schedule
- RTO (Recovery Time Objective): minutes (time to promote)

### Client Redirect

- Connection URL can be configured to auto-redirect on failover
- Uses **connection object** (account-level)
- Clients connect to connection URL, not directly to account
- On failover, connection redirects to new primary automatically

## 1.5 Data Sharing — Advanced

### Provider vs Consumer Responsibilities

| Provider | Consumer |
|----------|----------|
| Creates share | Creates database from share |
| Grants privileges on objects | Grants roles access to shared DB |
| Controls what's shared | Controls who in their account can access |
| Pays for storage | Pays for compute (queries) |
| Can revoke access anytime | Read-only access only |

### Secure Views in Data Sharing

- **MUST use secure views** when sharing filtered/transformed data
- Regular views expose definition via `GET_DDL()`
- Secure views hide definition from consumers
- Query optimizer may be less effective on secure views (cannot push predicates through)

```sql
CREATE SECURE VIEW shared_view AS
SELECT * FROM sensitive_table
WHERE region = CURRENT_ACCOUNT();  -- Multi-tenant pattern
```

### Sharing with Non-Snowflake Accounts (Reader Accounts)

- Provider creates a **Reader Account** for non-Snowflake consumers
- Provider pays for reader account compute AND storage
- Reader accounts are limited (no data loading, no sharing)
- Managed by the provider account

### Share Object Types

Can share:
- Tables (permanent, not temporary/transient)
- Secure views, secure materialized views
- Secure UDFs
- External tables

Cannot share:
- Temporary/transient tables
- Non-secure views
- Stages
- Pipes
- Streams/Tasks

### Data Exchange & Marketplace

| Feature | Data Exchange | Marketplace |
|---------|--------------|-------------|
| Audience | Invite-only group | Public |
| Listing types | Standard | Standard + Personalized |
| Discovery | Members only | Anyone with Snowflake account |
| Governance | Provider-managed membership | Snowflake-managed |

---

# DOMAIN 2: PERFORMANCE OPTIMIZATION (19%)

---

## 2.1 Query Profile — Deep Dive

### Reading the Query Profile

The Query Profile shows a **DAG (Directed Acyclic Graph)** of operators:

| Operator | What It Does |
|----------|-------------|
| TableScan | Reads micro-partitions from storage |
| Filter | Applies WHERE predicates |
| Join (Hash/Merge/Nested Loop) | Combines tables |
| Aggregate | GROUP BY / window functions |
| Sort | ORDER BY |
| Projection | SELECT column list |
| WindowFunction | OVER() clauses |
| Result | Final output |

### Key Metrics to Watch

| Metric | Warning Sign |
|--------|-------------|
| **Bytes spilled to local** | > 0 means warehouse too small for data volume |
| **Bytes spilled to remote** | Severe — data spilling to remote storage |
| **Partitions scanned vs total** | Low ratio = good pruning; high = bad |
| **Bytes sent over network** | High = data shuffling between nodes |
| **Percentage scanned from cache** | Higher = better (warehouse cache hit) |

### Spilling — Causes & Solutions

**What is spilling?**
- When intermediate results exceed warehouse memory
- Data "spills" to local SSD first, then to remote storage

**Spilling hierarchy:**
```
RAM (fastest) → Local SSD (slower) → Remote Storage (slowest)
```

**Solutions:**
1. Use a larger warehouse (more memory)
2. Reduce data processed (better filters, pruning)
3. Optimize joins (smaller table on build side)
4. Process less data (LIMIT, sampling)

### Exploding Joins

- Many-to-many joins can produce exponentially more rows
- Query Profile shows output rows >> input rows on Join operator
- Solution: check join keys for uniqueness, add filters before join

## 2.2 Materialized Views

### What Are Materialized Views?

- Pre-computed results stored physically
- Automatically maintained by Snowflake (serverless background process)
- Best for: queries on large tables with expensive aggregations where source data changes infrequently

```sql
CREATE MATERIALIZED VIEW mv_daily_sales AS
SELECT date, region, SUM(amount) as total
FROM sales
GROUP BY date, region;
```

### Materialized View Limitations

| Allowed | NOT Allowed |
|---------|-------------|
| JOINs (to a single table) | JOINs to multiple tables |
| Aggregations | HAVING clause |
| WHERE clause | UDFs |
| CLUSTER BY | LIMIT / ORDER BY |
| | Nested subqueries |
| | WINDOW functions |
| | Non-deterministic functions (CURRENT_TIMESTAMP) |

**Key Rules:**
- Source must be a single permanent table (not view, not external table)
- Cannot be built on top of another view
- Can query from a materialized view like a regular table
- Snowflake auto-rewrites queries to use MV when beneficial (even if you query the base table)
- Maintenance is serverless and billed separately
- Suspended MVs stop refreshing; resume to catch up

### Materialized View vs Regular View vs Table

| Feature | Table | View | Materialized View |
|---------|-------|------|-------------------|
| Stores data | Yes | No | Yes |
| Auto-updated | No | N/A (always current) | Yes (background) |
| Query performance | Fast | Depends on base query | Fast |
| Storage cost | Yes | No | Yes |
| Maintenance cost | No | No | Yes (serverless) |
| Can be clustered | Yes | No | Yes |

## 2.3 Multi-Cluster Warehouses

### Scaling Policy

| Policy | Behavior | Best For |
|--------|----------|----------|
| **Standard** (default) | Starts new cluster after 20s queue | Performance-sensitive workloads |
| **Economy** | Starts new cluster after 6 min queue | Cost-sensitive workloads |

### Scale Up vs Scale Out

| Strategy | How | When |
|----------|-----|------|
| **Scale Up** | Increase warehouse size (S→M→L) | Complex/slow queries |
| **Scale Out** | Add clusters (multi-cluster) | Many concurrent queries queuing |

**Key insight:** Scaling up helps individual query speed. Scaling out helps concurrency.

### Auto-Suspend & Auto-Resume

- `AUTO_SUSPEND = 300` — suspend after 5 min idle (seconds)
- `AUTO_RESUME = TRUE` — resume on query submission
- Minimum auto-suspend: 60 seconds (except 0 = never suspend)
- Credit billing: minimum 60 seconds per resume, then per-second

### Resource Monitors

```sql
CREATE RESOURCE MONITOR my_monitor
  WITH CREDIT_QUOTA = 1000
  FREQUENCY = MONTHLY
  START_TIMESTAMP = IMMEDIATELY
  TRIGGERS
    ON 75 PERCENT DO NOTIFY
    ON 90 PERCENT DO NOTIFY
    ON 100 PERCENT DO SUSPEND
    ON 110 PERCENT DO SUSPEND_IMMEDIATE;
```

| Action | Behavior |
|--------|----------|
| NOTIFY | Send alert, no action |
| SUSPEND | Stop new queries, let running finish |
| SUSPEND_IMMEDIATE | Kill running queries immediately |

- Can assign to account level OR individual warehouses
- Account-level monitor: controls total account spend
- Warehouse-level: controls per-warehouse spend
- A warehouse can have at most ONE resource monitor
- Only ACCOUNTADMIN can create resource monitors

## 2.4 Search Optimization Service — Advanced

### When SOS Helps (vs Clustering)

| Scenario | Use Clustering | Use SOS |
|----------|---------------|--------|
| Range queries on one column | Yes | Maybe |
| Point lookups (equality) | Maybe | Yes |
| Queries with LIKE/ILIKE | No | Yes |
| Queries on semi-structured (VARIANT) | Limited | Yes |
| Geospatial queries (GEOGRAPHY) | No | Yes |
| Queries with IN lists | Maybe | Yes |
| Substring search | No | Yes |

### SOS Configuration

```sql
-- Enable on specific columns
ALTER TABLE my_table ADD SEARCH OPTIMIZATION
  ON EQUALITY(col1, col2),
  ON SUBSTRING(col3),
  ON GEO(geo_col);

-- Check optimization status
SELECT SYSTEM$ESTIMATE_SEARCH_OPTIMIZATION_COSTS('my_table');
```

- Maintenance is serverless (separate billing)
- Not helpful for full table scans or queries returning large portions of data
- Works alongside clustering (complementary)
- Can be expensive for frequently changing tables

## 2.5 Query Tags & Warehouse Scheduling

### Query Tags

```sql
-- Set session-level tag
ALTER SESSION SET QUERY_TAG = 'ETL_DAILY_LOAD';

-- Set on specific query
SELECT /*+ QUERY_TAG='report_gen' */ * FROM my_table;
```

- Used for cost attribution, monitoring, debugging
- Visible in QUERY_HISTORY
- Can filter Account Usage views by QUERY_TAG
- Max 2000 characters

---

# DOMAIN 3: STORAGE AND DATA PROTECTION (14%)

---

## 3.1 Zero-Copy Cloning — Advanced

### What Gets Cloned?

```sql
CREATE TABLE cloned_table CLONE source_table;
CREATE SCHEMA cloned_schema CLONE source_schema;
CREATE DATABASE cloned_db CLONE source_db;
```

| Object | Clones children? | Clones data? | Clones privileges? |
|--------|-----------------|--------------|--------------------|
| Database | Yes (schemas, tables, etc.) | Yes (zero-copy) | NOT by default |
| Schema | Yes (tables, views, etc.) | Yes (zero-copy) | NOT by default |
| Table | N/A | Yes (zero-copy) | NOT by default |

**Key clone behaviors:**
- Cloning is metadata-only (instant, no storage cost initially)
- Storage cost incurs only when clone OR source is modified (divergence)
- Clones are independent objects — DML on one doesn't affect the other
- Can clone at a point in time: `CLONE ... AT(TIMESTAMP => ...)`
- Cloned tables inherit clustering keys
- Pipes, streams, tasks are NOT cloned (only within database clone)
- Stages are NOT cloned
- Internal named stages: cloned as empty
- External stages: reference same external location

### Clone + Time Travel

```sql
-- Clone a table as it was 1 hour ago
CREATE TABLE recovery_table CLONE my_table
  AT(OFFSET => -3600);

-- Clone from before a specific statement
CREATE TABLE recovery_table CLONE my_table
  BEFORE(STATEMENT => '01a23b45-0001-abcd-0000-00012345678');
```

### What Cannot Be Cloned

- External tables (only metadata, not external data)
- Internal stages (cloned empty)
- Pipes referencing internal stages won't work in clone
- Temporary tables cannot be cloned to permanent

## 3.2 Encryption & Data Protection

### Encryption Architecture

```
Data → File Key → Table Master Key → Account Master Key → Root Key
(each layer wraps the layer below — hierarchical key model)
```

**Key hierarchy:**
1. **Root Key** — Held by Snowflake (or customer with Tri-Secret Secure)
2. **Account Master Key** — Encrypts table master keys
3. **Table Master Key** — Encrypts file keys
4. **File Key** — Encrypts individual micro-partition files

### Tri-Secret Secure (Business Critical+)

- Customer provides their own key in cloud KMS (AWS KMS, Azure Key Vault, GCP KMS)
- Snowflake combines customer key + Snowflake key = composite master key
- If customer revokes their key, Snowflake cannot access data
- Available only on Business Critical edition or higher

### Periodic Rekeying

- Snowflake automatically rotates keys annually
- Periodic rekeying: re-encrypts actual data with new keys (not just key rotation)
- `ALTER ACCOUNT SET PERIODIC_DATA_REKEYING = TRUE;`
- Available on Enterprise edition+
- Different from key rotation (which only changes wrapping keys)

### End-to-End Encryption

| Feature | Standard | Enterprise | Business Critical |
|---------|----------|------------|-------------------|
| AES-256 encryption at rest | Yes | Yes | Yes |
| TLS 1.2 in transit | Yes | Yes | Yes |
| Automatic key rotation | Annual | Annual | Annual |
| Periodic rekeying | No | Yes | Yes |
| Tri-Secret Secure | No | No | Yes |
| AWS PrivateLink / Azure Private Link | No | No | Yes |

## 3.3 Time Travel & Fail-Safe — Advanced Considerations

### Data Retention by Edition & Table Type

| Table Type | Standard | Enterprise+ |
|-----------|----------|-------------|
| Permanent | 0-1 days | 0-90 days |
| Transient | 0-1 days | 0-1 days |
| Temporary | 0-1 days | 0-1 days |

### Fail-Safe

| Table Type | Fail-Safe Period |
|-----------|------------------|
| Permanent | 7 days (after Time Travel expires) |
| Transient | 0 days (NO fail-safe) |
| Temporary | 0 days (NO fail-safe) |

**Fail-Safe is NOT self-service:**
- Only Snowflake support can recover from Fail-Safe
- No guarantee of recovery (best-effort)
- Storage cost continues during Fail-Safe period

### Storage Cost Implications

```
Total storage = Active data + Time Travel data + Fail-Safe data
```

- Dropping a table: Time Travel still counts (until TT expires)
- After Time Travel: Fail-Safe kicks in (7 more days for permanent tables)
- Transient tables: cheaper (no fail-safe storage) but less protected
- Temporary tables: cheapest (no fail-safe, no TT beyond session)

### UNDROP

```sql
UNDROP TABLE my_table;    -- restores most recently dropped version
UNDROP SCHEMA my_schema;
UNDROP DATABASE my_db;
```

- Only works within Time Travel retention period
- If a new object with same name exists, must rename it first
- UNDROP restores child objects too (for DB/schema)
- Dropped objects count against Time Travel storage

## 3.4 Data Protection Patterns

### Network Policies

```sql
CREATE NETWORK POLICY my_policy
  ALLOWED_IP_LIST = ('192.168.1.0/24', '10.0.0.0/8')
  BLOCKED_IP_LIST = ('192.168.1.99');

-- Apply to account
ALTER ACCOUNT SET NETWORK_POLICY = my_policy;

-- Apply to specific user
ALTER USER john SET NETWORK_POLICY = my_policy;
```

- Blocked list takes precedence over allowed list
- User-level policy overrides account-level policy
- If no policy set, all IPs are allowed
- Only SECURITYADMIN+ can manage network policies

### Private Connectivity (Business Critical)

| Cloud | Feature | Purpose |
|-------|---------|--------|
| AWS | PrivateLink | Private network connection to Snowflake |
| Azure | Private Link | Same for Azure |
| GCP | Private Service Connect | Same for GCP |

- Traffic never traverses public internet
- Requires Business Critical edition
- Provides private endpoint within your VPC/VNet

---

# DOMAIN 4: DATA GOVERNANCE (14%)

---

## 4.1 Dynamic Data Masking

### What Is It?

A column-level security policy that masks data at query time based on the user's role.

```sql
-- Create masking policy
CREATE MASKING POLICY mask_ssn AS
  (val STRING) RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() IN ('HR_ADMIN', 'ACCOUNTADMIN') THEN val
    ELSE '***-**-' || RIGHT(val, 4)
  END;

-- Apply to column
ALTER TABLE employees MODIFY COLUMN ssn
  SET MASKING POLICY mask_ssn;
```

### Masking Policy Key Facts

| Aspect | Detail |
|--------|--------|
| Applied at | Column level |
| Evaluated at | Query time (not stored masked) |
| Input/output types | Must match (STRING→STRING, NUMBER→NUMBER) |
| Multiple columns | Same policy can apply to many columns |
| Per column | Only ONE masking policy per column |
| Conditional | Can use CURRENT_ROLE(), IS_ROLE_IN_SESSION(), etc. |
| Inheritance | Sub-roles inherit access of parent roles |
| Stacking | Cannot stack multiple masking policies on same column |

### Conditional Masking (Using another column)

```sql
CREATE MASKING POLICY conditional_mask AS
  (val STRING, visibility STRING) RETURNS STRING ->
  CASE
    WHEN visibility = 'PUBLIC' THEN val
    WHEN CURRENT_ROLE() IN ('ADMIN') THEN val
    ELSE '****'
  END;

-- Apply with conditional column
ALTER TABLE t MODIFY COLUMN email
  SET MASKING POLICY conditional_mask
  USING (email, access_level);
```

## 4.2 Row Access Policies

### What Is It?

A table/view-level policy that filters rows at query time based on the user's context.

```sql
-- Create row access policy
CREATE ROW ACCESS POLICY region_filter AS
  (region_val VARCHAR) RETURNS BOOLEAN ->
  CURRENT_ROLE() = 'ADMIN'
  OR region_val IN (
    SELECT region FROM role_region_mapping
    WHERE role_name = CURRENT_ROLE()
  );

-- Apply to table
ALTER TABLE sales ADD ROW ACCESS POLICY region_filter
  ON (region);
```

### Row Access Policy Key Facts

| Aspect | Detail |
|--------|--------|
| Applied at | Table or View level |
| Effect | Filters rows (invisible rows are hidden, not errored) |
| Returns | BOOLEAN (TRUE = visible, FALSE = hidden) |
| Per object | Only ONE row access policy per table/view |
| Mapping table | Commonly uses a lookup table for role→data mapping |
| Performance | Can impact query performance (evaluated per row) |
| Owner bypass | Table OWNER does NOT automatically bypass |
| ACCOUNTADMIN | Does NOT automatically bypass (must be coded in policy) |

### Masking Policy vs Row Access Policy

| Feature | Masking Policy | Row Access Policy |
|---------|---------------|-------------------|
| Scope | Column values | Entire rows |
| Returns | Same data type (masked value) | BOOLEAN (show/hide) |
| Applied to | Column | Table/View |
| Multiple per object | One per column (many columns) | One per table |
| Effect | Transforms values | Filters rows |

## 4.3 Object Tagging

### Tag Objects

```sql
-- Create a tag
CREATE TAG cost_center ALLOWED_VALUES = ('finance', 'engineering', 'marketing');
CREATE TAG pii_type ALLOWED_VALUES = ('email', 'ssn', 'phone', 'name');

-- Apply tag to objects
ALTER TABLE customers SET TAG cost_center = 'marketing';
ALTER TABLE customers MODIFY COLUMN email SET TAG pii_type = 'email';
```

### Tag Key Facts

| Aspect | Detail |
|--------|--------|
| Levels | Account, Database, Schema, Table, Column, Warehouse, etc. |
| Allowed values | Optional — can restrict valid tag values |
| Inheritance | Child objects inherit parent tags (schema→table→column) |
| Override | Child can override inherited tag value |
| Max tags per object | Unlimited (but practical limits) |
| Tag-based masking | Can associate masking policy with tag |
| Lineage | Tags propagate through `TAG_REFERENCES` view |

### Tag-Based Masking Policies

```sql
-- Associate masking policy with a tag
ALTER TAG pii_type SET MASKING POLICY mask_pii;
```

- When a column has a tag with an associated masking policy, the policy auto-applies
- Scales governance: tag once, mask everywhere
- Column-level policy overrides tag-based policy (more specific wins)

## 4.4 Data Classification

### Automatic Classification

```sql
-- Classify columns in a table
SELECT * FROM TABLE(
  SYSTEM$CLASSIFY('my_db.my_schema.my_table', {'auto_tag': true})
);
```

- Snowflake automatically identifies PII, sensitive data
- Categories: IDENTIFIER, QUASI_IDENTIFIER, SENSITIVE
- Semantic categories: NAME, EMAIL, PHONE, SSN, etc.
- Can auto-tag columns with classification results
- Uses sampling (not full table scan) for efficiency

### Access History & Object Dependencies

```sql
-- Who accessed what (Account Usage)
SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY
WHERE query_start_time > DATEADD('day', -7, CURRENT_TIMESTAMP());
```

- Tracks column-level access (reads and writes)
- Shows which columns were accessed, by whom, when
- Retained for 365 days
- Helps with compliance auditing and unused column identification

## 4.5 Secure Objects

### Secure Views

```sql
CREATE SECURE VIEW my_secure_view AS
SELECT col1, col2 FROM my_table WHERE condition;
```

**Secure vs Regular Views:**

| Feature | Regular View | Secure View |
|---------|-------------|-------------|
| Definition visible via GET_DDL | Yes | No (owner only) |
| Query optimizer | Full optimization | Limited (no predicate pushdown from outside) |
| Data sharing | Cannot share | Can share |
| Performance | Better | Slightly worse (optimizer restricted) |

### Secure UDFs

```sql
CREATE SECURE FUNCTION my_func(x INT) RETURNS INT
  AS 'x * 2';
```

- Definition hidden from non-owners
- Required for sharing UDFs via data sharing
- Same optimizer limitations as secure views

### Column-Level Security Summary

| Method | Use Case |
|--------|----------|
| Dynamic Masking | Show partial/masked data based on role |
| External Tokenization | Replace sensitive data with tokens (external service) |
| Secure Views | Hide entire columns + view logic |
| Row Access Policies | Hide entire rows based on role |

---

# DOMAIN 5: DATA TRANSFORMATION (25%)

---

## 5.1 Streams (Change Data Capture)

### What Are Streams?

Streams track DML changes (INSERT, UPDATE, DELETE) on a table, producing a change log.

```sql
CREATE STREAM my_stream ON TABLE my_table;
```

### Stream Types

| Type | Tracks | Use Case |
|------|--------|----------|
| **Standard** (default) | INSERT, UPDATE, DELETE | Full CDC |
| **Append-only** | INSERT only | Staging tables, logs |
| **Insert-only** | INSERT only (external tables) | External tables only |

```sql
CREATE STREAM append_stream ON TABLE logs APPEND_ONLY = TRUE;
CREATE STREAM ext_stream ON EXTERNAL TABLE ext_tbl INSERT_ONLY = TRUE;
```

### Stream Metadata Columns

| Column | Type | Meaning |
|--------|------|--------|
| METADATA$ACTION | VARCHAR | 'INSERT' or 'DELETE' |
| METADATA$ISUPDATE | BOOLEAN | TRUE if row is part of UPDATE |
| METADATA$ROW_ID | VARCHAR | Unique row identifier |

**How UPDATEs appear:**
- An UPDATE = DELETE (old row) + INSERT (new row)
- Both rows have `METADATA$ISUPDATE = TRUE`

### Stream Key Behaviors

| Behavior | Detail |
|----------|--------|
| Consumption | Stream is consumed (emptied) when used in DML within a transaction |
| Staleness | Stream becomes stale if not consumed within retention period |
| Retention | Tied to source table's Time Travel retention |
| Multiple streams | Multiple streams can exist on same table |
| Offset | Stream maintains its own offset (position in change log) |
| Transactional | Only committed changes appear in stream |
| SHOW command | `SHOW STREAMS` / `DESCRIBE STREAM` |

### Stream Staleness

- A stream goes **stale** if its offset falls behind the data retention period
- Stale stream cannot be consumed (must be recreated)
- Prevention: ensure stream is consumed regularly (within TT retention)
- `STALE_AFTER` column in `SHOW STREAMS` shows when it will go stale

## 5.2 Tasks

### What Are Tasks?

Tasks execute SQL statements on a schedule or trigger.

```sql
-- Time-based task
CREATE TASK daily_etl
  WAREHOUSE = compute_wh
  SCHEDULE = 'USING CRON 0 2 * * * America/New_York'
  AS
  INSERT INTO target SELECT * FROM my_stream WHERE METADATA$ACTION = 'INSERT';

-- Interval-based task
CREATE TASK every_5_min
  WAREHOUSE = compute_wh
  SCHEDULE = '5 MINUTE'
  AS
  CALL my_procedure();
```

### Task Trees (DAGs)

```
Root Task (has SCHEDULE)
├── Child Task A (AFTER root)
│   ├── Grandchild Task A1 (AFTER A)
│   └── Grandchild Task A2 (AFTER A)
└── Child Task B (AFTER root)
```

```sql
CREATE TASK child_a
  WAREHOUSE = compute_wh
  AFTER root_task
  AS
  INSERT INTO summary SELECT * FROM staging;
```

**Task tree rules:**
- Only ROOT task has a SCHEDULE
- Child tasks use `AFTER parent_task` clause
- A child can depend on multiple predecessors
- Max depth: no fixed limit, but practical limits apply
- All tasks in tree must be in same schema
- If root skips (WHEN condition false), entire tree skips

### WHEN Condition (Conditional Execution)

```sql
CREATE TASK conditional_task
  WAREHOUSE = compute_wh
  SCHEDULE = '5 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('my_stream')
  AS
  INSERT INTO target SELECT * FROM my_stream;
```

- `SYSTEM$STREAM_HAS_DATA()` — checks if stream has unconsumed changes
- WHEN is evaluated without consuming warehouse credits
- If FALSE, task (and all children) are skipped

### Task Management

```sql
-- Tasks are created SUSPENDED by default
ALTER TASK my_task RESUME;   -- Start scheduling
ALTER TASK my_task SUSPEND;  -- Stop scheduling

-- For task trees: suspend/resume from leaf to root (suspend) or root to leaf (resume)
-- Resume order: children first, then root
-- Suspend order: root first, then children
```

### Serverless Tasks

```sql
CREATE TASK serverless_task
  USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
  SCHEDULE = '5 MINUTE'
  AS
  SELECT 1;
```

- No warehouse specified = serverless
- Snowflake manages compute (auto-scales)
- Billed based on actual compute used
- Good for variable workloads

## 5.3 Stored Procedures

### Languages Supported

| Language | Use Case |
|----------|----------|
| SQL (Scripting) | Simple procedural logic |
| JavaScript | Legacy, complex logic |
| Python | Data science, Snowpark |
| Java | Enterprise integration |
| Scala | Snowpark |

### SQL Scripting Stored Procedures

```sql
CREATE PROCEDURE process_data(table_name VARCHAR)
  RETURNS VARCHAR
  LANGUAGE SQL
  EXECUTE AS CALLER  -- or OWNER
  AS
  BEGIN
    LET row_count INTEGER := 0;
    SELECT COUNT(*) INTO :row_count FROM IDENTIFIER(:table_name);
    RETURN 'Processed ' || :row_count || ' rows';
  END;
```

### Caller's Rights vs Owner's Rights

| Feature | EXECUTE AS CALLER | EXECUTE AS OWNER |
|---------|-------------------|------------------|
| Runs with privileges of | Caller | Procedure owner |
| Access to caller's objects | Yes | No (only owner's) |
| Security | Less controlled | More controlled |
| Default | No | **YES (default)** |
| Session context | Caller's | Owner's |

**Key distinction:**
- Owner's rights (default): procedure uses owner's privileges regardless of who calls it
- Caller's rights: procedure uses the calling user's privileges
- Use caller's rights when procedure should operate on caller's data
- Use owner's rights for controlled access (like a secure API)

### Anonymous Blocks (SQL Scripting)

```sql
BEGIN
  LET x := 10;
  LET y := 20;
  RETURN x + y;
END;
```

- No CREATE needed — execute procedural code directly
- Useful for one-time scripting tasks
- Same SQL scripting syntax as stored procedures

## 5.4 User-Defined Functions (UDFs)

### UDF Types

| Type | Returns | Use Case |
|------|---------|----------|
| Scalar UDF | Single value per row | Transformations |
| UDTF (Table Function) | Table (multiple rows) | Row generation/expansion |
| UDAF (Aggregate) | Single value per group | Custom aggregations |

### Scalar UDF Example

```sql
CREATE FUNCTION fahrenheit_to_celsius(temp_f FLOAT)
  RETURNS FLOAT
  LANGUAGE SQL
  AS
  $$ (temp_f - 32) * 5.0 / 9.0 $$;

-- Usage
SELECT fahrenheit_to_celsius(212);  -- Returns 100.0
```

### Python UDTF (Table Function)

```sql
CREATE FUNCTION split_words(sentence VARCHAR)
  RETURNS TABLE(word VARCHAR)
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.8'
  HANDLER = 'SplitWords'
  AS $$
class SplitWords:
    def process(self, sentence):
        for word in sentence.split():
            yield (word,)
$$;

-- Usage
SELECT * FROM TABLE(split_words('hello world'));
```

### UDF vs Stored Procedure

| Feature | UDF | Stored Procedure |
|---------|-----|------------------|
| Called from | SELECT, WHERE, etc. | CALL statement |
| Returns | Value(s) for each row | Single value |
| Can execute DML | NO | YES |
| Can execute DDL | NO | YES |
| Used in expressions | YES | NO |
| Transaction control | NO | YES |
| Side effects | NO (must be deterministic-ish) | YES |

### External Functions

```sql
CREATE EXTERNAL FUNCTION sentiment(text VARCHAR)
  RETURNS VARIANT
  API_INTEGRATION = my_api_integration
  AS 'https://my-api-gateway.com/sentiment';
```

- Calls external HTTP endpoint (AWS Lambda, Azure Function, etc.)
- Requires API Integration object
- Scalar function semantics (one input → one output)
- Batched: Snowflake sends rows in batches for efficiency
- Latency: significantly slower than native UDFs (network round-trip)
- Use for: ML inference, external lookups, services not available in Snowflake

## 5.5 Semi-Structured Data — Advanced

### FLATTEN Function

```sql
-- Flatten an array
SELECT
  f.value::STRING AS item
FROM my_table,
  LATERAL FLATTEN(input => my_array_col) f;

-- Flatten nested JSON
SELECT
  f.value:name::STRING AS name,
  f.value:age::INT AS age
FROM my_table,
  LATERAL FLATTEN(input => json_col:employees) f;
```

### FLATTEN Output Columns

| Column | Description |
|--------|-------------|
| SEQ | Sequence number (unique per input row) |
| KEY | Key for maps, index for arrays |
| PATH | Path to the element |
| INDEX | Array index (NULL for objects) |
| VALUE | The element value |
| THIS | The original input to FLATTEN |

### LATERAL

- `LATERAL` allows a subquery to reference columns from preceding tables
- Required with FLATTEN to correlate with each row
- Like a correlated subquery in FROM clause

```sql
-- LATERAL join example
SELECT t.id, l.item
FROM my_table t,
  LATERAL (SELECT value AS item FROM TABLE(FLATTEN(t.items))) l;
```

### VARIANT, OBJECT, ARRAY

| Type | Stores | Access Pattern |
|------|--------|---------------|
| VARIANT | Any JSON value | `:key` or `[index]` |
| OBJECT | Key-value pairs | `:key` |
| ARRAY | Ordered list | `[index]` |

```sql
-- Type casting from VARIANT
SELECT
  data:name::STRING,
  data:age::INT,
  data:scores[0]::FLOAT,
  data:address:city::STRING
FROM json_table;
```

### PARSE_JSON vs TRY_PARSE_JSON

- `PARSE_JSON(string)` — errors if invalid JSON
- `TRY_PARSE_JSON(string)` — returns NULL if invalid JSON

## 5.6 Advanced SQL Features

### MERGE Statement

```sql
MERGE INTO target t
USING source s ON t.id = s.id
WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET t.name = s.name, t.updated = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (id, name) VALUES (s.id, s.name);
```

- Combines INSERT, UPDATE, DELETE in one statement
- Atomic operation (all or nothing)
- Common pattern with streams: MERGE stream data into target
- Multiple WHEN MATCHED clauses allowed (evaluated in order)

### Recursive CTEs

```sql
WITH RECURSIVE org_chart AS (
  -- Anchor: top-level managers
  SELECT employee_id, name, manager_id, 1 AS level
  FROM employees WHERE manager_id IS NULL
  
  UNION ALL
  
  -- Recursive: employees under each manager
  SELECT e.employee_id, e.name, e.manager_id, oc.level + 1
  FROM employees e
  JOIN org_chart oc ON e.manager_id = oc.employee_id
)
SELECT * FROM org_chart;
```

- Used for hierarchical/tree data (org charts, BOM, categories)
- Must have anchor member + recursive member
- Connected by UNION ALL
- Terminates when recursive member returns no rows

### QUALIFY Clause

```sql
-- Get latest record per customer (deduplication)
SELECT *
FROM orders
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) = 1;
```

- Filters on window function results
- Like HAVING but for window functions
- Eliminates need for subquery wrapper
- Snowflake-specific (not standard SQL)

### GENERATOR & SEQUENCES

```sql
-- Generate rows
SELECT SEQ4() AS row_num
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

-- Sequences
CREATE SEQUENCE my_seq START = 1 INCREMENT = 1;
SELECT my_seq.NEXTVAL;  -- Gets next value
```

### Table Literals & VALUES

```sql
SELECT * FROM VALUES
  (1, 'Alice', 30),
  (2, 'Bob', 25),
  (3, 'Carol', 35)
  AS t(id, name, age);
```

## 5.7 Snowpark (Python/Java/Scala)

### What Is Snowpark?

- DataFrame API that pushes computation to Snowflake (server-side)
- Write Python/Java/Scala, executes as SQL in Snowflake
- No data movement — processing happens in Snowflake

### Key Snowpark Concepts

```python
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, sum, avg

session = Session.builder.configs(connection_params).create()

# DataFrame operations (lazy — not executed until action)
df = session.table("sales")
result = df.filter(col("year") == 2024) \
           .group_by("region") \
           .agg(sum("amount").alias("total_sales")) \
           .sort(col("total_sales").desc())

# Action — triggers execution
result.show()
result.collect()  # Returns list of Row objects
pdf = result.to_pandas()  # Convert to pandas DataFrame
```

### Lazy vs Eager Evaluation

| Type | Operations | When Executed |
|------|-----------|---------------|
| Lazy (transformations) | filter, select, group_by, join, sort | Not until action |
| Eager (actions) | show(), collect(), count(), to_pandas() | Immediately |

### Snowpark Stored Procedures

```sql
CREATE PROCEDURE snowpark_proc(table_name STRING)
  RETURNS STRING
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.8'
  PACKAGES = ('snowflake-snowpark-python')
  HANDLER = 'main'
  AS $$
def main(session, table_name):
    df = session.table(table_name)
    count = df.count()
    return f"Table has {count} rows"
$$;
```

### Snowpark vs Traditional SQL

| Feature | Snowpark | SQL |
|---------|----------|-----|
| Language | Python/Java/Scala | SQL |
| Execution | Server-side (Snowflake) | Server-side |
| Best for | Complex transformations, ML | Set-based operations |
| Debugging | IDE, local testing | Query Profile |
| Reusability | Libraries, classes | Views, procedures |

---

# QUICK REFERENCE — KEY DIFFERENCES TO REMEMBER

---

## Edition-Based Features

| Feature | Standard | Enterprise | Business Critical |
|---------|----------|------------|-------------------|
| Time Travel (max) | 1 day | 90 days | 90 days |
| Materialized Views | No | Yes | Yes |
| Multi-cluster Warehouse | No | Yes | Yes |
| Search Optimization | No | Yes | Yes |
| Column-level Security | No | Yes | Yes |
| Row Access Policies | No | Yes | Yes |
| Object Tagging | No | Yes | Yes |
| Data Classification | No | Yes | Yes |
| Periodic Rekeying | No | Yes | Yes |
| Tri-Secret Secure | No | No | Yes |
| PrivateLink | No | No | Yes |
| Database Failover | No | No | Yes |
| Data Sharing (provider) | Yes | Yes | Yes |

## Serverless Features (No Warehouse Needed)

| Feature | Billed As |
|---------|----------|
| Snowpipe | Serverless credits (per-file) |
| Serverless Tasks | Serverless credits (compute time) |
| Materialized View maintenance | Serverless credits |
| Search Optimization maintenance | Serverless credits |
| Automatic Clustering | Serverless credits |
| Replication | Data transfer + serverless |
| Query Acceleration Service | Serverless credits |

## Common "Gotcha" Questions

| Question | Answer |
|----------|--------|
| Default ON_ERROR for Snowpipe? | SKIP_FILE |
| Default ON_ERROR for COPY INTO? | ABORT_STATEMENT |
| Can you UNDROP a transient table? | Yes (within its 0-1 day retention) |
| Does fail-safe apply to transient tables? | NO |
| Default task state on creation? | SUSPENDED |
| Can a stream go stale? | Yes (if not consumed within retention) |
| Can you share a regular view? | NO (must be SECURE view) |
| Who pays for compute in data sharing? | Consumer |
| Who pays for reader account compute? | Provider |
| Does ACCOUNTADMIN bypass row access policy? | NO (unless coded in policy) |
| Default procedure execution rights? | OWNER's rights |
| Can UDFs execute DML? | NO (only procedures can) |
| Can you clone a temporary table to permanent? | NO |
| How long does COPY metadata last? | 64 days |
| Max Time Travel for transient table? | 1 day (any edition) |
| QUALIFY is standard SQL? | NO (Snowflake extension) |